# Ingesta del archivo results.json 
1. Leer todos los archivos desde el folder resultados usando API spark dataframe reader 
1. Definir y garantizar el esquema 
1. Agregar columnas metadata 
    - Archivo fuente
    - Ingesta Timestamp
1. Escribir a la tabla bronze delta  

In [0]:
%run ../../../proyecto_final_formula1/PrepAmb/02.Ayuda_Bronze

In [0]:
%run ../../../proyecto_final_formula1/PrepAmb/03.Configuracion_Ambiente

In [0]:
# Definir el archivo fuente y tabla 
archivo_fuente = f"{ruta}/results"
nombre_tabla = f"{catalogo}.{esquema_bronze}.results"

#### 1 - Leer el archivo json usando API dataframe reader

In [0]:
# Define the schema
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType, DateType

esquema_results = StructType([
    StructField("date", DateType()),
    StructField("raceName", StringType()),
    StructField("round", IntegerType()),
    StructField("season", IntegerType()),
    StructField("url", StringType()),
    StructField("constructorId", StringType()),
    StructField("driverId", StringType()),
    StructField("grid", IntegerType()),
    StructField("laps", IntegerType()),
    StructField("number", IntegerType()),
    StructField("points", FloatType()),
    StructField("position", IntegerType()),
    StructField("positionText", StringType()),
    StructField("status", StringType())
])

In [0]:
# Leer los archivos de resultados
df_results = (
    spark.read
       .format('json')
       .schema(esquema_results)
       .option('mode', 'FAILFAST')
       .load(archivo_fuente)
)

#### 2 - Agregar columans de metadatos
- Archivo fuente
- Ingesta Timestamp

In [0]:
df_results_final = add_ingesta_metadata(df_results)

#### 3 - Escribir a la tabla bronze delta

In [0]:
(
    df_results_final
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(nombre_tabla)
)

In [0]:
display(spark.table(nombre_tabla))